In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Continuous corners：固定独立开发比较
仅运行固定4个新prompt/seed × CG/G/U × 6条件：72观察，最多216次当前内容统计评分，`science_denominator=0`。每观察一次raw几何输出，共享pre、分别强制rounded/continuous post。原始统计、失败与固定行保留。

**逐格运行，不要 Run all。** 首个unit为18观察/最多54评分；查看实测时间后，手动运行其余3个unit（最多162评分）。不更换失败样本、不重试或扩大名单。生成含每unit两次同seed SD3.5调用及生成器内部host LF测量；它们独立记录在生成耗时中，不属于216次检测统计。

旧tau仅描述性参考，派生早退结果使用已有pre/post，不增加模型调用。identity强制post下降不等于生产最终判决下降。黑填充为候选协议，reflect为压力条件；G/U分开解释。不是正式校准、FPR或论文测试。

兼容当前代码的已加载pipeline/assets优先复用；旧版本geometry模块已载入时需先重启runtime，防止混用枚举与转换实现。新建runtime使用原生产工厂，不限定A100。需要Colab Secrets `CEG_WM_ROOT_KEY`，新建模型还需要`HF_TOKEN`。

In [ ]:
from pathlib import Path
import sys, subprocess, json, time
EXACT='c7da6c9f273cebfaf18269b77e90cb38fd3b8c77'
REPO=Path('/content/ceg-wm-continuous-dev-c7da6c9')
OUTPUT=Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V2/continuous-corners-dev-v1')
if OUTPUT.exists(): raise FileExistsError('已有输出，保留原结果；不要自动覆盖或重跑。')
loaded=sys.modules.get('cegwm.geometry_v7.contracts')
if loaded is not None and not hasattr(loaded,'syncseal_raw_to_public_continuous'):
    raise RuntimeError('旧geometry模块已载入；请重启runtime，再逐格运行此notebook。')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','BlindDetection-V2','--single-branch','https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('保留已有代码改动。')
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
sys.path.insert(0,str(REPO/'src'))
from diagnostics.continuous_corners_v1.runner import Session, validate_runtime
print({'code':EXACT,'observations':72,'max_score_calls':216,'output':str(OUTPUT)})

In [ ]:
from google.colab import userdata
import torch
from cegwm.shared.keys import normalize_detection_key, public_key_digest
from cegwm.protocol.content_chain import CONTENT_CHAIN_PUBLIC_KEY_DIGEST
try: dev_key=normalize_detection_key(userdata.get('CEG_WM_ROOT_KEY'))
except Exception: raise RuntimeError('请启用有效Colab Secret CEG_WM_ROOT_KEY') from None
if public_key_digest(dev_key)!=CONTENT_CHAIN_PUBLIC_KEY_DIGEST:
    raise ValueError('Secret与原内容链密钥不一致；尚未初始化模型。')
started=time.perf_counter()
candidates=[]
for prefix in ('dev_','renderer_',''):
    candidates.append((prefix+'pipeline/assets',globals().get(prefix+'pipeline'),globals().get(prefix+'assets')))
for name in ('dev_session','renderer_session','session'):
    candidate=globals().get(name)
    if candidate is not None:
        candidates.append((name,getattr(candidate,'pipeline',None),getattr(candidate,'assets',None)))
dev_pipeline=dev_assets=None
for label,pipe,assets_candidate in candidates:
    if pipe is None or assets_candidate is None: continue
    try: validate_runtime(pipe,assets_candidate,dev_key)
    except (TypeError,ValueError,AttributeError): continue
    dev_pipeline,dev_assets,asset_source=pipe,assets_candidate,label
    break
if dev_assets is None:
    try: token=userdata.get('HF_TOKEN')
    except Exception: raise RuntimeError('请启用Colab Secret HF_TOKEN') from None
    from experiments.run_blind_detection_v1 import build_production_runtime,load_runtime_config
    runtime_root=Path('/content/continuous-dev-runtime')
    runtime_root.mkdir(exist_ok=True)
    dev_pipeline,dev_assets=build_production_runtime(REPO,load_runtime_config(REPO),hf_token=token,runtime_root=runtime_root)
    del token
    asset_source='existing production factory, fresh runtime'
dev_session=Session(dev_pipeline,dev_assets,dev_key,OUTPUT)
initialization=dict(code=EXACT,initialization_seconds=time.perf_counter()-started,asset_source=asset_source,
    torch=torch.__version__,cuda=torch.cuda.is_available(),gpu=torch.cuda.get_device_name() if torch.cuda.is_available() else None,
    geometry_device=str(dev_assets.geometry_backend.device),science_denominator=0)
(OUTPUT/'initialization.json').write_text(json.dumps(initialization,indent=2))
print(initialization)

## 首个 unit：运行后停下查看耗时与错误
18观察、最多54次评分。此格不会继续剩余3个unit。

In [ ]:
pilot=dev_session.pilot()
(OUTPUT/'pilot_summary.json').write_text(json.dumps(pilot,indent=2))
print(json.dumps(pilot,indent=2))

## 手动继续其余三个 unit
查看上格的生成、同步嵌入、几何、评分和总耗时后，再手动运行此格。剩余54观察、最多162次评分；估时只按首unit外推。

In [ ]:
summary=dev_session.remaining()
print(json.dumps(summary,indent=2))

In [ ]:
import pandas as pd
table=pd.read_csv(OUTPUT/'paired.csv')
display(table)
print({'rows':len(table),'unique_observations':len(table.drop_duplicates(['unit','arm','condition'])),
       'threshold_role':'historical descriptive only','science_denominator':0})